# Fine-tune `BAAI/bge-large-en-v1.5` on your own text files

This notebook fine-tunes the `bge-large-en-v1.5` sentence-embedding model using `sentence-transformers`, given a **list of paths to text files** containing your training pairs.

## Assumption about your input files
Fine-tuning an embedding model requires *pairs* (or triplets) of related texts, not just raw unlabeled text. This notebook assumes:

- Each file is a **plain text file, tab-separated**, one training example per line:
  ```
  anchor text<TAB>positive text
  anchor text<TAB>positive text<TAB>hard negative text   (optional 3rd column)
  ```
- `anchor` = e.g. a query, a clause, a sentence
- `positive` = a text that should be embedded *close* to the anchor (e.g. a matching clause, a paraphrase, a relevant passage)
- `negative` (optional) = a text that should be embedded *far* from the anchor (a hard negative). If omitted, in-batch negatives are used automatically.

**If your files are in a different format** (JSONL, CSV, single-column, etc.), you only need to edit the `parse_file()` function in the Data Loading section below — everything downstream works off the list of `InputExample` objects it produces.

## What this notebook does
1. Installs dependencies
2. Loads and parses your text files into training pairs
3. Splits into train / eval
4. Fine-tunes `bge-large-en-v1.5` with `MultipleNegativesRankingLoss`
5. Evaluates and saves the fine-tuned model (downloadable, or savable to Google Drive)

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers==3.* datasets accelerate

## 2. Provide your list of text file paths

If your files live on Google Drive, mount it first. Otherwise, upload the files directly to the Colab session and reference their paths.

In [ ]:
# --- OPTION A: mount Google Drive (uncomment if your files are on Drive) ---
# from google.colab import drive
# drive.mount('/content/drive')

# --- OPTION B: upload files directly to this Colab session ---
# from google.colab import files
# uploaded = files.upload()
# file_paths = list(uploaded.keys())

# --- EDIT THIS: list of paths to your text files ---
file_paths = [
    "/content/train_pairs_1.txt",
    "/content/train_pairs_2.txt",
]

print(f"{len(file_paths)} file(s) provided")

## 3. Load and parse the files into training examples

Edit `parse_file()` here if your file format differs from the tab-separated assumption above.

In [ ]:
from sentence_transformers import InputExample
import random

def parse_file(path):
    """Parse one tab-separated file into a list of InputExample objects.
    Each line: anchor<TAB>positive[<TAB>negative]
    Blank lines and lines starting with # are skipped.
    """
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.rstrip("\n")
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) == 2:
                anchor, positive = parts
                examples.append(InputExample(texts=[anchor.strip(), positive.strip()]))
            elif len(parts) == 3:
                anchor, positive, negative = parts
                examples.append(InputExample(texts=[anchor.strip(), positive.strip(), negative.strip()]))
            else:
                print(f"  Skipping malformed line {line_num} in {path}: expected 2 or 3 tab-separated fields, got {len(parts)}")
    return examples

all_examples = []
for path in file_paths:
    file_examples = parse_file(path)
    print(f"{path}: {len(file_examples)} examples")
    all_examples.extend(file_examples)

print(f"\nTotal examples loaded: {len(all_examples)}")
assert len(all_examples) > 0, "No training examples were loaded — check your file paths and format."

In [ ]:
# Train / eval split
random.seed(42)
random.shuffle(all_examples)

eval_fraction = 0.1  # 10% held out for evaluation
split_idx = max(1, int(len(all_examples) * (1 - eval_fraction)))
train_examples = all_examples[:split_idx]
eval_examples = all_examples[split_idx:]

print(f"Train examples: {len(train_examples)}")
print(f"Eval examples:  {len(eval_examples)}")

## 4. Load the base model

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected. In Colab, go to Runtime > Change runtime type > select a GPU.")

model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)

## 5. Set up training

Uses `MultipleNegativesRankingLoss`, the standard contrastive loss for fine-tuning retrieval/embedding models. It works with (anchor, positive) pairs, using other in-batch positives as negatives, and automatically also uses the third column as a hard negative when present.

Adjust `num_epochs`, `batch_size`, and `warmup_ratio` below as needed. Keep `batch_size` as large as your GPU memory allows — larger batches give more in-batch negatives, which usually improves quality for this loss.

In [ ]:
# --- Training hyperparameters (edit as needed) ---
batch_size = 16
num_epochs = 3
warmup_ratio = 0.1
learning_rate = 2e-5
output_path = "/content/bge-large-en-v1.5-finetuned"

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = int(len(train_dataloader) * num_epochs * warmup_ratio)
print(f"Training steps per epoch: {len(train_dataloader)}")
print(f"Warmup steps: {warmup_steps}")

## 6. (Optional) Set up an evaluator on the held-out examples

This gives you a similarity-based score during training so you can see if the model is improving. It treats each held-out pair as a positive (label=1); this is a rough signal, not a full retrieval benchmark.

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = None
if len(eval_examples) > 0:
    eval_anchors = [ex.texts[0] for ex in eval_examples]
    eval_positives = [ex.texts[1] for ex in eval_examples]
    eval_labels = [1.0] * len(eval_examples)
    evaluator = EmbeddingSimilarityEvaluator(
        eval_anchors, eval_positives, eval_labels, name="eval-pairs"
    )
else:
    print("No eval examples available — skipping evaluator.")

## 7. Fine-tune

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": learning_rate},
    output_path=output_path,
    show_progress_bar=True,
    save_best_model=True if evaluator is not None else False,
)

print(f"\nFine-tuned model saved to: {output_path}")

## 8. Quick sanity check on the fine-tuned model

In [ ]:
from sentence_transformers.util import cos_sim

finetuned_model = SentenceTransformer(output_path, device=device)

# Edit these with a couple of examples relevant to your task
sample_texts = [
    "Example anchor text here",
    "Example text that should be similar",
    "Example text that should be dissimilar",
]
embeddings = finetuned_model.encode(sample_texts, normalize_embeddings=True)

print("Similarity (anchor vs. similar):   ", cos_sim(embeddings[0], embeddings[1]).item())
print("Similarity (anchor vs. dissimilar):", cos_sim(embeddings[0], embeddings[2]).item())

## 9. Save your model for later use

Pick whichever option fits your workflow.

In [ ]:
# --- OPTION A: zip and download directly ---
import shutil
from google.colab import files

zip_path = shutil.make_archive("bge-large-en-v1.5-finetuned", "zip", output_path)
files.download(zip_path)

In [ ]:
# --- OPTION B: save to Google Drive instead (uncomment; requires drive.mount() above) ---
# import shutil
# drive_dest = "/content/drive/MyDrive/bge-large-en-v1.5-finetuned"
# shutil.copytree(output_path, drive_dest, dirs_exist_ok=True)
# print(f"Model copied to {drive_dest}")

## 10. (Optional) Push to the Hugging Face Hub

In [ ]:
# from huggingface_hub import login
# login()  # paste your HF token when prompted
# finetuned_model.push_to_hub("your-username/bge-large-en-v1.5-finetuned")